<a href="https://colab.research.google.com/github/SeoYun-Y/thinfiler-credit-scoring/blob/track-b%2F01-target-variable/notebooks/track-b/track_b_01_%ED%83%80%EA%B2%9F%EB%B3%80%EC%88%98_%EA%B2%80%EC%A6%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Track B (Sheet11, 통신카드CB결합정보) 타겟변수 확정 과정

이 노트북은 Track B의 씬파일러 정의를 전제로, **지도학습에 쓸 타겟변수를 찾는 전체 탐색 과정**을 순서대로 담고 있습니다.

**최종 결론 먼저 요약**: 씬파일러(약 4.1만 명) 세그먼트 **내부**에서는 어떤 방법으로도 유의미한 연체 타겟을 만들 수 없음을 5가지 방식으로 확인했습니다. 대신 **신용이력 보유군(약 28.9만 명)에서 `PYE_D1011000C`(1년 내 연체발생, 해제포함)를 타겟으로 학습**하고, 씬파일러에는 정답 없이 스코어링만 적용하는 구조로 최종 확정했습니다.

---

## 1. 데이터 위치 확인

드라이브 마운트 후 폴더 구조를 확인합니다. 원본 8개 분기 CSV, 공식 코드북, 이전 변수스캔 결과 위치를 파악합니다.## 드라이브 마운트

In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 폴더 구조 확인

In [35]:
import os
import pandas as pd
os.listdir('/content/drive/MyDrive/')

['캡스톤디자인 2조',
 'runs',
 'data',
 'Colab Notebooks',
 'Gallery',
 '[서비스 소개서 양식] 한국관광공사 2026 관광 프롬프톤_팀명_제출자명 (사본 만들어서 사용하세요)의 사본.gslides',
 '변수사전_Sheet09_개인CB정보.gdoc',
 '씬파일러_재정의_코랩분석_팀공유용.gdoc',
 '변수사전_Sheet11_통신카드CB결합.gdoc',
 '멀티캠퍼스 데이터분석_신용평가 데이터']

In [36]:
base_path = '/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터'
import os
os.listdir(base_path)

['093-117_금융 합성데이터_데이터구성상세.xlsx',
 '09.개인_CB정보',
 '변수스캔결과',
 'data',
 '통신카드CB_씬파일러.csv']

In [37]:
os.listdir(f'{base_path}/data')

['개인CB_씬파일러.csv', '통신카드CB_씬파일러.csv']

In [38]:
os.listdir(f'{base_path}/변수스캔결과')

['track_a_전수스캔_20250713.csv', 'track_b_전수스캔_20250713.csv']

**확인 결과**: 원본 8개 분기 파일은 `통신카드CB_씬파일러.csv` 폴더 안에 있고, 코드북은 `093-117_금융 합성데이터_데이터구성상세.xlsx`. 이후 모든 셀에서 이 경로를 `scan_path`로 사용합니다.

## 2. 코드북 구조 파악 — Sheet11 전체 필드 확인

공식 코드북에서 시트 목록을 먼저 보고, `11.통신카드CB 결합정보` 시트(738개 필드)를 전부 불러옵니다.

**목적**: Track A(Sheet09)처럼 PERF1~4 같은 이진 연체 라벨이 Track B에도 있는지 직접 확인.

In [39]:
codebook_path = f'{base_path}/093-117_금융 합성데이터_데이터구성상세.xlsx'

xls = pd.ExcelFile(codebook_path)
print(xls.sheet_names)

['1.회원 정보', '2.신용 정보', '3.승인매출 정보', '4.청구입금 정보', '5.잔액 정보', '6.채널 정보', '7.마케팅 정보', '8.성과 정보', '9.개인CB 정보', '10.기업CB 정보', '11.통신카드CB 결합정보', '12.금융상품정보', '별첨1. 개인기업CB 코드정보', '별첨2. 결합CB 코드정보']


In [40]:
sheet11 = pd.read_excel(codebook_path, sheet_name='11.통신카드CB 결합정보')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
print(sheet11.shape)
sheet11

(738, 5)


,No,필드명,설명,코드여부,타입
0,1,BASE_YM,기준년월,N,Char
1,2,CUST_ID,고객식별자,N,Char
2,3,SEX,성별,Y,Char
3,4,AGE,연령대,Y,Char
4,5,JB_TP,직업군,Y,Char
5,6,HOME_ADM,거주지시도,Y,Char
6,7,COM_ADM,근무지시도,Y,Char
7,8,HIGHEND_CD1,하이엔드대상자코드1,Y,Char
8,9,HIGHEND_CD2,하이엔드대상자코드2,Y,Char
9,10,HIGHEND_CD3,하이엔드대상자코드3,Y,Char


**확인 결과**: Sheet11에는 이진 연체 라벨(PERF류)이 없습니다. 대신 연체와 관련된 필드는 다음 8개뿐입니다.

| 필드명 | 의미 |
|---|---|
| `PYE_SC0000000` | KCB Score (연속형) |
| `PYE_D10110001/003/006` | 연체건수, 발생(해제포함), 1/3/6개월 |
| `PYE_D1011000C` | 연체건수, 발생(해제포함), 1년(12개월) |
| `PYE_D10210D00` | 연체건수, 미해제 |
| `PYE_D10232000` | 총연체금액, 미해제 |
| `PYE_MAX_DLQ_DAY` | 최장연체일수, 미해제 |

모두 `PYE_` 접두사 — **전년말 기준 고정 스냅샷**(분기 내 값이 갱신되지 않음, 아래에서 직접 검증).

## 3. 사전 확인 — CUST_ID 추적 가능 여부 및 씬파일러 정의

타겟을 연도 간 비교로 설계하려면, 먼저 CUST_ID가 연도를 넘어 동일인을 추적 가능한지 확인해야 합니다.

※ 씬파일러 정의(5개 필드 전부 0: `PYE_C1M210000`, `PYE_C18233003~005`, `PYE_L10210000`)는 팀 합의로 이미 확정된 상태입니다(비율 13.37%/2021, 12.53%/2022). 이 노트북은 이 정의를 전제로 타겟변수만 탐색합니다.

In [41]:
scan_path = f'{base_path}/통신카드CB_씬파일러.csv'
os.listdir(scan_path)

['202103_통신카드CB결합.csv',
 '202106_통신카드CB결합.csv',
 '202109_통신카드CB결합.csv',
 '202112_통신카드CB결합.csv',
 '202203_통신카드CB결합.csv',
 '202206_통신카드CB결합.csv',
 '202209_통신카드CB결합.csv',
 '202212_통신카드CB결합.csv']

In [42]:
import pandas as pd

id_2021 = pd.read_csv(f'{scan_path}/202103_통신카드CB결합.csv', usecols=['CUST_ID'])
id_2022 = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=['CUST_ID'])

set_2021 = set(id_2021['CUST_ID'])
set_2022 = set(id_2022['CUST_ID'])

overlap = set_2021 & set_2022
print(f"2021 고객 수: {len(set_2021):,}")
print(f"2022 고객 수: {len(set_2022):,}")
print(f"교집합: {len(overlap):,}")
print(f"2021 대비 교집합 비율: {len(overlap)/len(set_2021)*100:.1f}%")
print(f"2022 대비 교집합 비율: {len(overlap)/len(set_2022)*100:.1f}%")

2021 고객 수: 330,000
2022 고객 수: 330,000
교집합: 330,000
2021 대비 교집합 비율: 100.0%
2022 대비 교집합 비율: 100.0%


**확인 결과**: 2021년·2022년 CUST_ID가 100% 동일(33만 명) — 연도 간 동일인 추적 가능. 타겟을 연도 간 비교로 설계할 수 있는 전제 조건 통과.

## 4. 타겟 후보 시도 ① — 연도 간 변화(PYE_D 계열)

`PYE_` 필드가 전년말 고정 스냅샷이라는 걸 이용해, **2021년 값 → 2022년 값의 변화**를 연체 발생 신호로 볼 수 있는지 탐색.

먼저 같은 연도 내 분기(Q1 vs Q2)가 정말 동일한지, 연도 간(2021 vs 2022)은 얼마나 다른지 확인합니다.

In [43]:
cols = ['CUST_ID', 'PYE_D10110001', 'PYE_D10110003', 'PYE_D10110006',
         'PYE_D1011000C', 'PYE_D10210D00', 'PYE_D10232000']

d_2021 = pd.read_csv(f'{scan_path}/202103_통신카드CB결합.csv', usecols=cols)
d_2022 = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=cols)

merged = d_2021.merge(d_2022, on='CUST_ID', suffixes=('_2021', '_2022'))

# 값이 같은 사람 비율 확인 (완전히 고정값이면 100%에 가까울 것)
for col in ['PYE_D10110001', 'PYE_D10110003', 'PYE_D10110006',
            'PYE_D1011000C', 'PYE_D10210D00', 'PYE_D10232000']:
    same_ratio = (merged[f'{col}_2021'] == merged[f'{col}_2022']).mean()
    print(f"{col}: 동일값 비율 {same_ratio*100:.1f}%")

PYE_D10110001: 동일값 비율 99.8%
PYE_D10110003: 동일값 비율 99.4%
PYE_D10110006: 동일값 비율 99.2%
PYE_D1011000C: 동일값 비율 98.6%
PYE_D10210D00: 동일값 비율 99.8%
PYE_D10232000: 동일값 비율 99.5%


In [44]:
d_2021_q2 = pd.read_csv(f'{scan_path}/202106_통신카드CB결합.csv', usecols=cols)
merged_q = d_2021.merge(d_2021_q2, on='CUST_ID', suffixes=('_Q1', '_Q2'))

for col in ['PYE_D10110001', 'PYE_D10110003']:
    same_ratio = (merged_q[f'{col}_Q1'] == merged_q[f'{col}_Q2']).mean()
    print(f"{col} (Q1 vs Q2, 같은 연도): 동일값 비율 {same_ratio*100:.1f}%")

PYE_D10110001 (Q1 vs Q2, 같은 연도): 동일값 비율 100.0%
PYE_D10110003 (Q1 vs Q2, 같은 연도): 동일값 비율 100.0%


In [45]:
# 변화의 방향 확인 - 0에서 양수로 바뀐 경우(신규 연체 발생)를 타겟 후보로
target_col = 'PYE_D10110001'  # 1개월내 연체건수, 가장 민감한 지표로 시작

merged['changed'] = merged[f'{target_col}_2021'] != merged[f'{target_col}_2022']
merged['new_delinquency'] = (merged[f'{target_col}_2021'] == 0) & (merged[f'{target_col}_2022'] > 0)

print(f"전체 변화 인원: {merged['changed'].sum():,}명")
print(f"신규 연체 발생(0→양수): {merged['new_delinquency'].sum():,}명")
print(f"신규 연체 발생 비율: {merged['new_delinquency'].mean()*100:.3f}%")

# 값 분포도 확인
print(merged[f'{target_col}_2021'].value_counts().head())
print(merged[f'{target_col}_2022'].value_counts().head())

전체 변화 인원: 797명
신규 연체 발생(0→양수): 370명
신규 연체 발생 비율: 0.112%
PYE_D10110001_2021
0.0    328808
1.0      1192
Name: count, dtype: int64
PYE_D10110001_2022
0.0    328865
1.0      1135
Name: count, dtype: int64


In [46]:
# 씬파일러 정의 필드까지 포함해서 다시 읽기
thin_cols = ['CUST_ID', 'PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004',
             'PYE_C18233005', 'PYE_L10210000',
             'PYE_D10110001', 'PYE_D1011000C']  # 1개월, 1년 두 버전 같이 확인

d21 = pd.read_csv(f'{scan_path}/202103_통신카드CB결합.csv', usecols=thin_cols)
d22 = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=thin_cols)

merged2 = d21.merge(d22, on='CUST_ID', suffixes=('_2021', '_2022'))

# 씬파일러 정의: 2021년 시점 기준 5개 필드 전부 0
thin_mask = (
    (merged2['PYE_C1M210000_2021'] == 0) &
    (merged2['PYE_C18233003_2021'] == 0) &
    (merged2['PYE_C18233004_2021'] == 0) &
    (merged2['PYE_C18233005_2021'] == 0) &
    (merged2['PYE_L10210000_2021'] == 0)
)

thin_df = merged2[thin_mask]
print(f"씬파일러 인원: {len(thin_df):,}명")

for target_col in ['PYE_D10110001', 'PYE_D1011000C']:
    new_delq = (thin_df[f'{target_col}_2021'] == 0) & (thin_df[f'{target_col}_2022'] > 0)
    print(f"{target_col} 신규 연체(씬파일러 내): {new_delq.sum()}명 ({new_delq.mean()*100:.3f}%)")

씬파일러 인원: 44,110명
PYE_D10110001 신규 연체(씬파일러 내): 6명 (0.014%)
PYE_D1011000C 신규 연체(씬파일러 내): 2명 (0.005%)


In [47]:
delq_cols = ['CUST_ID', 'PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004',
             'PYE_C18233005', 'PYE_L10210000',
             'PYE_D10110001', 'PYE_D10110003', 'PYE_D10110006',
             'PYE_D1011000C', 'PYE_D10210D00', 'PYE_D10232000']

d21 = pd.read_csv(f'{scan_path}/202103_통신카드CB결합.csv', usecols=delq_cols)
d22 = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=delq_cols)

merged3 = d21.merge(d22, on='CUST_ID', suffixes=('_2021', '_2022'))

thin_mask = (
    (merged3['PYE_C1M210000_2021'] == 0) &
    (merged3['PYE_C18233003_2021'] == 0) &
    (merged3['PYE_C18233004_2021'] == 0) &
    (merged3['PYE_C18233005_2021'] == 0) &
    (merged3['PYE_L10210000_2021'] == 0)
)
thin_df = merged3[thin_mask]

delq_fields = ['PYE_D10110001', 'PYE_D10110003', 'PYE_D10110006',
               'PYE_D1011000C', 'PYE_D10210D00']

# 6개 건수 필드 중 하나라도 증가했으면 "악화"로 표시
worsened = pd.Series(False, index=thin_df.index)
for col in delq_fields:
    worsened |= (thin_df[f'{col}_2022'] > thin_df[f'{col}_2021'])

print(f"씬파일러 인원: {len(thin_df):,}명")
print(f"건수 필드 중 하나라도 악화: {worsened.sum()}명 ({worsened.mean()*100:.3f}%)")

# 금액 필드 증가도 별도 확인
amt_worsened = thin_df['PYE_D10232000_2022'] > thin_df['PYE_D10232000_2021']
print(f"연체금액 증가: {amt_worsened.sum()}명 ({amt_worsened.mean()*100:.3f}%)")

# 둘 중 하나라도
any_worsened = worsened | amt_worsened
print(f"둘 중 하나라도 악화: {any_worsened.sum()}명 ({any_worsened.mean()*100:.3f}%)")

씬파일러 인원: 44,110명
건수 필드 중 하나라도 악화: 109명 (0.247%)
연체금액 증가: 4명 (0.009%)
둘 중 하나라도 악화: 113명 (0.256%)


**확인 결과**:
- 같은 연도 내 분기(Q1 vs Q2)는 **100% 동일** → 8개 분기가 실제로는 2개 스냅샷(2021, 2022)의 4회 중복 저장임을 재확인
- 씬파일러 중 6개 연체필드 중 하나라도 '악화'된 사람: **113명(0.256%)**

**판정: 폐기.** 표본 규모가 통계적으로 유의미한 학습에 쓰기엔 너무 작음.

## 5. 타겟 후보 시도 ② — KCB Score(`PYE_SC0000000`) 등급화

기획서 원안이었던 KCB Score 등급화를 재검토합니다. 팀원이 작성한 별도 문서에서 이미 폐기 근거(86.41% 변동, 방향 역전)가 제시됐던 부분이라, 직접 재현하여 검증합니다.

In [48]:
score_cols = ['CUST_ID', 'PYE_SC0000000',
              'PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004',
              'PYE_C18233005', 'PYE_L10210000']

d = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=score_cols)

# 1. 전체 분포 확인 (결측/0 비율)
print(d['PYE_SC0000000'].describe())
print(f"0점 비율: {(d['PYE_SC0000000']==0).mean()*100:.2f}%")
print(f"결측 비율: {d['PYE_SC0000000'].isna().mean()*100:.2f}%")

# 2. 씬파일러 vs 신용이력 보유군 나눠서 스코어 분포 비교
thin_mask = (
    (d['PYE_C1M210000']==0) & (d['PYE_C18233003']==0) &
    (d['PYE_C18233004']==0) & (d['PYE_C18233005']==0) &
    (d['PYE_L10210000']==0)
)

print("\n씬파일러 스코어 분포:")
print(d.loc[thin_mask, 'PYE_SC0000000'].describe())

print("\n신용이력 보유군(비-씬파일러) 스코어 분포:")
print(d.loc[~thin_mask, 'PYE_SC0000000'].describe())

count    330000.000000
mean        838.404952
std         200.778973
min           0.000000
25%         764.000000
50%         911.000000
75%         979.000000
max        1000.000000
Name: PYE_SC0000000, dtype: float64
0점 비율: 2.67%
결측 비율: 0.00%

씬파일러 스코어 분포:
count    41333.000000
mean       693.517480
std        209.322225
min          0.000000
25%        692.000000
50%        760.000000
75%        801.000000
max       1000.000000
Name: PYE_SC0000000, dtype: float64

신용이력 보유군(비-씬파일러) 스코어 분포:
count    288667.000000
mean        859.150772
std         190.720922
min           0.000000
25%         808.000000
50%         932.000000
75%         987.000000
max        1000.000000
Name: PYE_SC0000000, dtype: float64


In [49]:
cols = ['CUST_ID', 'PYE_SC0000000',
        'PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004', 'PYE_C18233005']

d21 = pd.read_csv(f'{scan_path}/202103_통신카드CB결합.csv', usecols=cols)
d22 = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=cols)

merged_sc = d21.merge(d22, on='CUST_ID', suffixes=('_2021', '_2022'))

# 변동 비율
changed = merged_sc['PYE_SC0000000_2021'] != merged_sc['PYE_SC0000000_2022']
print(f"KCB Score 변동 비율: {changed.mean()*100:.2f}%")

# 변동 폭 분포도 같이 확인 (얼마나 크게 변했는지)
diff = (merged_sc['PYE_SC0000000_2022'] - merged_sc['PYE_SC0000000_2021']).abs()
print(diff.describe())

KCB Score 변동 비율: 86.41%
count    330000.000000
mean         32.594785
std          72.272911
min           0.000000
25%           5.000000
50%          17.000000
75%          37.000000
max        1000.000000
dtype: float64


In [50]:
delq_cols = ['CUST_ID', 'PYE_SC0000000', 'PYE_D1011000C',  # 1년내발생 연체건수
             'PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004', 'PYE_C18233005']

d21b = pd.read_csv(f'{scan_path}/202103_통신카드CB결합.csv', usecols=delq_cols)
d22b = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=delq_cols)

def thin_mask(df):
    return (
        (df['PYE_C1M210000']==0) & (df['PYE_C18233003']==0) &
        (df['PYE_C18233004']==0) & (df['PYE_C18233005']==0)
    )

def delq_rate_compare(df, year_label):
    thin = thin_mask(df)
    thin_delq = (df.loc[thin, 'PYE_D1011000C'] > 0).mean()
    general_delq = (df.loc[~thin, 'PYE_D1011000C'] > 0).mean()
    print(f"[{year_label}] 씬파일러 연체율: {thin_delq*100:.3f}% / 일반 연체율: {general_delq*100:.3f}% / 배율: {thin_delq/general_delq if general_delq>0 else float('nan'):.2f}배")

delq_rate_compare(d21b, "2021 (202103 기준, KCB SCORE=0 정의 아님 — 보유건수 기준)")
delq_rate_compare(d22b, "2022 (202203 기준)")

[2021 (202103 기준, KCB SCORE=0 정의 아님 — 보유건수 기준)] 씬파일러 연체율: 1.693% / 일반 연체율: 4.416% / 배율: 0.38배
[2022 (202203 기준)] 씬파일러 연체율: 1.506% / 일반 연체율: 4.096% / 배율: 0.37배


In [51]:
def thin_mask_score(df):
    return df['PYE_SC0000000'] == 0

def delq_rate_compare_score(df, year_label):
    thin = thin_mask_score(df)
    thin_delq = (df.loc[thin, 'PYE_D1011000C'] > 0).mean()
    general_delq = (df.loc[~thin, 'PYE_D1011000C'] > 0).mean()
    print(f"[{year_label}] SCORE=0 기준 씬파일러 인원: {thin.sum()}명")
    print(f"  씬파일러 연체율: {thin_delq*100:.3f}% / 일반 연체율: {general_delq*100:.3f}% / 배율: {thin_delq/general_delq if general_delq>0 else float('nan'):.2f}배")

delq_rate_compare_score(d21b, "2021")
delq_rate_compare_score(d22b, "2022")

[2021] SCORE=0 기준 씬파일러 인원: 9381명
  씬파일러 연체율: 3.038% / 일반 연체율: 4.044% / 배율: 0.75배
[2022] SCORE=0 기준 씬파일러 인원: 8814명
  씬파일러 연체율: 3.687% / 일반 연체율: 3.737% / 배율: 0.99배


**확인 결과**:
- 씬파일러(41,333명) KCB Score 평균 693.5 vs 신용이력보유군 859.2 — 얼핏 변별력 있어 보였음
- 그러나 **연도간 변동 비율 86.41%** — 매우 불안정
- SCORE=0 정의 기준 연체율 배율: 2021년 0.75배 → 2022년 0.99배 (거의 1:1로 수렴, 변별력 소실 추세)
- 팀원 문서가 지적한 '방향 역전'까지 포함하면 시계열 안정성 자체가 담보되지 않음

**판정: 폐기.** 값 자체가 신뢰할 수 없을 정도로 불안정함.

## 6. 타겟 후보 시도 ③ — `PYE_MAX_DLQ_DAY`(최장연체일수, 미해제) 단순 이진화

'최장연체일수 > 0'을 그대로 이진 타겟으로 쓸 수 있는지 검토합니다.

In [52]:
cols = ['CUST_ID', 'PYE_MAX_DLQ_DAY',
        'PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004', 'PYE_C18233005']

d2 = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=cols)

# 전체 분포
print(d2['PYE_MAX_DLQ_DAY'].describe())
print(f"0(정상) 비율: {(d2['PYE_MAX_DLQ_DAY']==0).mean()*100:.3f}%")
print(f"1이상(연체) 비율: {(d2['PYE_MAX_DLQ_DAY']>0).mean()*100:.3f}%")

# 4개 필드 기준 씬파일러 재정의 (문서와 동일하게)
thin_mask_4 = (
    (d2['PYE_C1M210000']==0) & (d2['PYE_C18233003']==0) &
    (d2['PYE_C18233004']==0) & (d2['PYE_C18233005']==0)
)
print(f"\n4개필드 기준 씬파일러 인원: {thin_mask_4.sum():,}명 ({thin_mask_4.mean()*100:.2f}%)")

thin_dlq = d2.loc[thin_mask_4, 'PYE_MAX_DLQ_DAY']
print(f"씬파일러 내 연체(1일이상) 비율: {(thin_dlq>0).mean()*100:.3f}%")
print(f"씬파일러 내 연체 인원: {(thin_dlq>0).sum()}명")

count    330000.000000
mean          0.655870
std          16.120486
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         941.000000
Name: PYE_MAX_DLQ_DAY, dtype: float64
0(정상) 비율: 99.634%
1이상(연체) 비율: 0.366%

4개필드 기준 씬파일러 인원: 45,870명 (13.90%)
씬파일러 내 연체(1일이상) 비율: 0.179%
씬파일러 내 연체 인원: 82명


In [53]:
cols = ['CUST_ID', 'PYE_MAX_DLQ_DAY', 'PYE_C1M210000', 'PYE_C18233003',
         'PYE_C18233004', 'PYE_C18233005']

d21 = pd.read_csv(f'{scan_path}/202103_통신카드CB결합.csv', usecols=cols)
d22 = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=cols)

def thin_v2(df):
    return (df['PYE_C1M210000']==0) & (df['PYE_C18233003']==0) & \
           (df['PYE_C18233004']==0) & (df['PYE_C18233005']==0)

for label, df in [('2021', d21), ('2022', d22)]:
    thin = thin_v2(df)
    dlq_thin = (df.loc[thin, 'PYE_MAX_DLQ_DAY'] > 0).mean() * 100
    dlq_gen = (df.loc[~thin, 'PYE_MAX_DLQ_DAY'] > 0).mean() * 100
    print(f"[{label}] 전체 연체율: {(df['PYE_MAX_DLQ_DAY']>0).mean()*100:.3f}% / "
          f"씬파일러(보유건수기준): {dlq_thin:.3f}% / 신용이력보유군: {dlq_gen:.3f}%")

# 값 자체의 연도간 안정성 (KCB Score처럼 요동치는지)
merged = d21.merge(d22, on='CUST_ID', suffixes=('_21','_22'))
changed = merged['PYE_MAX_DLQ_DAY_21'] != merged['PYE_MAX_DLQ_DAY_22']
print(f"\nPYE_MAX_DLQ_DAY 연도간 변동 비율: {changed.mean()*100:.2f}%")
print(f"(참고: KCB Score는 86.41%였음)")

[2021] 전체 연체율: 0.396% / 씬파일러(보유건수기준): 0.181% / 신용이력보유군: 0.433%
[2022] 전체 연체율: 0.366% / 씬파일러(보유건수기준): 0.179% / 신용이력보유군: 0.396%

PYE_MAX_DLQ_DAY 연도간 변동 비율: 0.45%
(참고: KCB Score는 86.41%였음)


In [54]:
cols = ['CUST_ID', 'PYE_MAX_DLQ_DAY', 'PYE_D10210D00',
        'PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004', 'PYE_C18233005',
        'PYE_L10210000']

d = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=cols)

# 1. 노출 수준(카드+대출 총 보유건수)과 연체율의 관계
d['total_exposure'] = d['PYE_C1M210000'] + d['PYE_L10210000']

exposure_bins = pd.cut(d['total_exposure'], bins=[-1,0,1,2,5,100],
                        labels=['0건','1건','2건','3~5건','6건+'])
print("=== 노출 수준별 연체율(PYE_MAX_DLQ_DAY>0) ===")
print(d.groupby(exposure_bins)['PYE_MAX_DLQ_DAY'].apply(lambda x: (x>0).mean()*100))

# 2. 4개필드(카드만) 기준 씬파일러 중, 실제로 대출(L10210000)은 있는 사람이 얼마나 되는지
thin_card_only = (
    (d['PYE_C1M210000']==0) & (d['PYE_C18233003']==0) &
    (d['PYE_C18233004']==0) & (d['PYE_C18233005']==0)
)
print(f"\n카드기준 씬파일러: {thin_card_only.sum()}명")
print(f"이 중 대출(PYE_L10210000>0) 있는 사람: {(d.loc[thin_card_only,'PYE_L10210000']>0).sum()}명")

# 3. 진짜 완전 무노출(카드0 + 대출0) 집단만 다시 확인
true_zero_exposure = thin_card_only & (d['PYE_L10210000']==0)
print(f"\n완전 무노출(카드0+대출0): {true_zero_exposure.sum()}명")
dlq_true_zero = (d.loc[true_zero_exposure, 'PYE_MAX_DLQ_DAY'] > 0).sum()
print(f"이 중 연체이력(>0) 있는 사람: {dlq_true_zero}명 ({dlq_true_zero/true_zero_exposure.sum()*100:.4f}%)")

# 4. 완전 무노출인데 연체 있는 사람들 상세 확인 (해지 후 미해제 가설 검증)
anomaly = d[true_zero_exposure & (d['PYE_MAX_DLQ_DAY']>0)]
print(f"\n=== 이상 케이스 상세 ({len(anomaly)}명) ===")
print(anomaly[['PYE_MAX_DLQ_DAY','PYE_D10210D00','PYE_C1M210000','PYE_L10210000']].describe())

=== 노출 수준별 연체율(PYE_MAX_DLQ_DAY>0) ===
total_exposure
0건      0.028946
1건      0.199236
2건      0.202865
3~5건    0.495957
6건+     0.535308
Name: PYE_MAX_DLQ_DAY, dtype: float64

카드기준 씬파일러: 45870명
이 중 대출(PYE_L10210000>0) 있는 사람: 4537명

완전 무노출(카드0+대출0): 41333명
이 중 연체이력(>0) 있는 사람: 9명 (0.0218%)

=== 이상 케이스 상세 (9명) ===
       PYE_MAX_DLQ_DAY  PYE_D10210D00  PYE_C1M210000  PYE_L10210000
count         9.000000            9.0            9.0            9.0
mean        221.555556            1.0            0.0            0.0
std         261.236058            0.0            0.0            0.0
min          12.000000            1.0            0.0            0.0
25%          12.000000            1.0            0.0            0.0
50%          13.000000            1.0            0.0            0.0
75%         417.000000            1.0            0.0            0.0
max         682.000000            1.0            0.0            0.0


/tmp/ipykernel_3246/525195259.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(d.groupby(exposure_bins)['PYE_MAX_DLQ_DAY'].apply(lambda x: (x>0).mean()*100))


**확인 결과**:
- 4개필드 기준 씬파일러(45,870명) 중 연체이력 82명 → 이 중 4,537명은 사실 **대출은 보유**하고 있었음(정의가 카드만 봐서 놓친 케이스)
- 대출까지 포함한 **완전 무노출(41,333명)로 좁히면 연체이력 단 9명(0.0218%)**
- 노출 수준(카드+대출 보유건수)과 연체율이 완벽히 단조 증가: 0건 0.03% → 1건 0.20% → 2건 0.20% → 3~5건 0.50% → 6건+ 0.54%
- 9명 전원 `PYE_D10210D00`(미해제건수)=1 → '카드·대출은 현재 0이지만 과거 계좌 하나가 미해제 연체로 남은' 예외 케이스

**판정: 폐기.** 씬파일러 정의 필드와 이 타겟이 **동일 시점(전년말) 스냅샷**이라 구조적으로 항등식에 가까움 — 노출이 0이면 연체할 대상 자체가 없음.

## 7. 타겟 후보 시도 ④ — 노출(신규대출) 발생 후 결과 관측

Track A의 '노출조건부 모델링' 논리를 이식: 씬파일러가 이후 신규대출(`CRDT_LN_BAL_NEW`, `HOUS_LN_BAL_NEW`)을 받았는지 먼저 보고, 그 다음 시점 연체를 타겟으로 삼는 forward 구조를 시도합니다.

In [55]:
cols_def = ['CUST_ID', 'PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004',
            'PYE_C18233005', 'PYE_L10210000']
cols_exp = ['CUST_ID', 'HOUS_LN_BAL_NEW', 'CRDT_LN_BAL_NEW']
cols_tgt = ['CUST_ID', 'PYE_MAX_DLQ_DAY']

# 1. 정의: 202103(2020년말 기준) 씬파일러
d_def = pd.read_csv(f'{scan_path}/202103_통신카드CB결합.csv', usecols=cols_def)
thin = (
    (d_def['PYE_C1M210000']==0) & (d_def['PYE_C18233003']==0) &
    (d_def['PYE_C18233004']==0) & (d_def['PYE_C18233005']==0) &
    (d_def['PYE_L10210000']==0)
)
thin_ids = set(d_def.loc[thin, 'CUST_ID'])
print(f"2020년말 기준 씬파일러: {len(thin_ids):,}명")

# 2. 노출: 2021년 4개 분기 중 하나라도 신규대출 발생 여부 확인
exposed_ids = set()
for q in ['202103', '202106', '202109', '202112']:
    d_exp = pd.read_csv(f'{scan_path}/{q}_통신카드CB결합.csv', usecols=cols_exp+['CUST_ID'])
    d_exp = d_exp[d_exp['CUST_ID'].isin(thin_ids)]
    new_loan = (d_exp['HOUS_LN_BAL_NEW']=='Y') | (d_exp['CRDT_LN_BAL_NEW']=='Y')
    exposed_ids |= set(d_exp.loc[new_loan, 'CUST_ID'])
    print(f"[{q}] 확인 완료, 누적 노출 인원: {len(exposed_ids)}")

print(f"\n2021년 중 신규대출 발생한 (원래)씬파일러: {len(exposed_ids):,}명")

# 3. 타겟: 2021년말 기준 연체 여부
d_tgt = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=cols_tgt)
d_tgt_exposed = d_tgt[d_tgt['CUST_ID'].isin(exposed_ids)]
dlq = (d_tgt_exposed['PYE_MAX_DLQ_DAY'] > 0).sum()
print(f"\n노출군 중 2021년말 연체이력 있는 사람: {dlq}명 / {len(d_tgt_exposed)}명 ({dlq/len(d_tgt_exposed)*100:.2f}%)")

2020년말 기준 씬파일러: 44,110명
[202103] 확인 완료, 누적 노출 인원: 0
[202106] 확인 완료, 누적 노출 인원: 0
[202109] 확인 완료, 누적 노출 인원: 0
[202112] 확인 완료, 누적 노출 인원: 0

2021년 중 신규대출 발생한 (원래)씬파일러: 0명

노출군 중 2021년말 연체이력 있는 사람: 0명 / 0명 (nan%)


/tmp/ipykernel_3246/3300629049.py:31: RuntimeWarning: invalid value encountered in scalar divide
  print(f"\n노출군 중 2021년말 연체이력 있는 사람: {dlq}명 / {len(d_tgt_exposed)}명 ({dlq/len(d_tgt_exposed)*100:.2f}%)")


In [56]:
d_check = pd.read_csv(f'{scan_path}/202106_통신카드CB결합.csv',
                       usecols=['CUST_ID', 'HOUS_LN_BAL_NEW', 'CRDT_LN_BAL_NEW'])

print("=== HOUS_LN_BAL_NEW 값 분포 ===")
print(d_check['HOUS_LN_BAL_NEW'].value_counts(dropna=False))

print("\n=== CRDT_LN_BAL_NEW 값 분포 ===")
print(d_check['CRDT_LN_BAL_NEW'].value_counts(dropna=False))

=== HOUS_LN_BAL_NEW 값 분포 ===
HOUS_LN_BAL_NEW
0    330000
Name: count, dtype: int64

=== CRDT_LN_BAL_NEW 값 분포 ===
CRDT_LN_BAL_NEW
0    324330
1      5670
Name: count, dtype: int64


In [57]:
cols_def = ['CUST_ID', 'PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004',
            'PYE_C18233005', 'PYE_L10210000']
cols_exp = ['CUST_ID', 'HOUS_LN_BAL_NEW', 'CRDT_LN_BAL_NEW']
cols_tgt = ['CUST_ID', 'PYE_MAX_DLQ_DAY']

# 1. 정의: 202103(2020년말 기준) 씬파일러 - 이미 확인됨 44,110명
d_def = pd.read_csv(f'{scan_path}/202103_통신카드CB결합.csv', usecols=cols_def)
thin = (
    (d_def['PYE_C1M210000']==0) & (d_def['PYE_C18233003']==0) &
    (d_def['PYE_C18233004']==0) & (d_def['PYE_C18233005']==0) &
    (d_def['PYE_L10210000']==0)
)
thin_ids = set(d_def.loc[thin, 'CUST_ID'])
print(f"2020년말 기준 씬파일러: {len(thin_ids):,}명")

# 2. 노출: 2021년 4개 분기 중 하나라도 신규대출 발생(=1) 여부 확인 (수정: 0/1 정수 기준)
exposed_ids = set()
for q in ['202103', '202106', '202109', '202112']:
    d_exp = pd.read_csv(f'{scan_path}/{q}_통신카드CB결합.csv', usecols=cols_exp+['CUST_ID'])
    d_exp = d_exp[d_exp['CUST_ID'].isin(thin_ids)]
    new_loan = (d_exp['HOUS_LN_BAL_NEW']==1) | (d_exp['CRDT_LN_BAL_NEW']==1)
    exposed_ids |= set(d_exp.loc[new_loan, 'CUST_ID'])
    print(f"[{q}] 확인 완료, 누적 노출 인원: {len(exposed_ids)}")

print(f"\n2021년 중 신규대출 발생한 (원래)씬파일러: {len(exposed_ids):,}명")

# 3. 타겟: 2021년말 기준 연체 여부
d_tgt = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=cols_tgt)
d_tgt_exposed = d_tgt[d_tgt['CUST_ID'].isin(exposed_ids)]
dlq = (d_tgt_exposed['PYE_MAX_DLQ_DAY'] > 0).sum()
print(f"\n노출군 중 2021년말 연체이력 있는 사람: {dlq}명 / {len(d_tgt_exposed)}명 ({dlq/len(d_tgt_exposed)*100:.2f}%)")

# 참고: 비교 기준선 - 전체 씬파일러(노출 무관) 대비 연체율
d_tgt_all_thin = d_tgt[d_tgt['CUST_ID'].isin(thin_ids)]
dlq_all = (d_tgt_all_thin['PYE_MAX_DLQ_DAY'] > 0).sum()
print(f"참고 - 전체 씬파일러(노출무관) 연체율: {dlq_all}명 / {len(d_tgt_all_thin)}명 ({dlq_all/len(d_tgt_all_thin)*100:.3f}%)")

2020년말 기준 씬파일러: 44,110명
[202103] 확인 완료, 누적 노출 인원: 0
[202106] 확인 완료, 누적 노출 인원: 0
[202109] 확인 완료, 누적 노출 인원: 0
[202112] 확인 완료, 누적 노출 인원: 0

2021년 중 신규대출 발생한 (원래)씬파일러: 0명


/tmp/ipykernel_3246/2549882713.py:31: RuntimeWarning: invalid value encountered in scalar divide
  print(f"\n노출군 중 2021년말 연체이력 있는 사람: {dlq}명 / {len(d_tgt_exposed)}명 ({dlq/len(d_tgt_exposed)*100:.2f}%)")



노출군 중 2021년말 연체이력 있는 사람: 0명 / 0명 (nan%)
참고 - 전체 씬파일러(노출무관) 연체율: 7명 / 44110명 (0.016%)


In [58]:
d_check2 = pd.read_csv(f'{scan_path}/202103_통신카드CB결합.csv',
                        usecols=['CUST_ID', 'CRDT_LN_BAL_NEW', 'HOUS_LN_BAL_NEW'])

# 씬파일러만 걸러서 분포 재확인
d_thin_check = d_check2[d_check2['CUST_ID'].isin(thin_ids)]
print("=== 씬파일러 내 CRDT_LN_BAL_NEW 분포 ===")
print(d_thin_check['CRDT_LN_BAL_NEW'].value_counts(dropna=False))

print("\n=== 씬파일러 내 HOUS_LN_BAL_NEW 분포 ===")
print(d_thin_check['HOUS_LN_BAL_NEW'].value_counts(dropna=False))

# 비교: 비-씬파일러(신용이력 보유군)에서는 어떻게 나오는지
d_nonthin_check = d_check2[~d_check2['CUST_ID'].isin(thin_ids)]
print("\n=== 신용이력 보유군 내 CRDT_LN_BAL_NEW 분포 (비교용) ===")
print(d_nonthin_check['CRDT_LN_BAL_NEW'].value_counts(dropna=False))

=== 씬파일러 내 CRDT_LN_BAL_NEW 분포 ===
CRDT_LN_BAL_NEW
0    44110
Name: count, dtype: int64

=== 씬파일러 내 HOUS_LN_BAL_NEW 분포 ===
HOUS_LN_BAL_NEW
0    44110
Name: count, dtype: int64

=== 신용이력 보유군 내 CRDT_LN_BAL_NEW 분포 (비교용) ===
CRDT_LN_BAL_NEW
0    280355
1      5535
Name: count, dtype: int64


In [59]:
exposed_ids_2y = set()
for q in ['202103','202106','202109','202112','202203','202206','202209','202212']:
    d_exp = pd.read_csv(f'{scan_path}/{q}_통신카드CB결합.csv', usecols=cols_exp+['CUST_ID'])
    d_exp = d_exp[d_exp['CUST_ID'].isin(thin_ids)]
    new_loan = (d_exp['HOUS_LN_BAL_NEW']==1) | (d_exp['CRDT_LN_BAL_NEW']==1)
    exposed_ids_2y |= set(d_exp.loc[new_loan, 'CUST_ID'])

print(f"2년(2021~2022) 내 신규대출 발생한 씬파일러: {len(exposed_ids_2y):,}명")

2년(2021~2022) 내 신규대출 발생한 씬파일러: 0명


**확인 결과**:
- 첫 시도는 값 인코딩을 잘못 짚어(`=='Y'`) 0명으로 나왔던 버그 → 실제 값은 0/1 정수임을 확인
- 정정 후에도 1년(2021) 관측: 신규대출 발생 씬파일러 **0명**
- 씬파일러 44,110명 전원 `CRDT_LN_BAL_NEW`=0, `HOUS_LN_BAL_NEW`=0 (신용이력보유군에서는 5,535명이 1로 확인됨 — 필드 자체는 정상 작동)
- 관측기간을 2년(2021~2022)으로 늘려도 **0명**

**판정: 폐기.** 씬파일러는 관측기간(최대 2년) 내내 신용활동 자체를 시작한 사례가 없음 — 데이터 자체가 씬파일러 정의를 강하게 재확인해주는 결과.

## 8. 1차 잠정 확정 — 신용이력 보유군/씬파일러 분리 구조

시도 ①~④가 모두 '씬파일러 세그먼트 **내부**에서' 타겟을 찾으려 했기 때문에 막혔다는 걸 확인한 뒤, 기획서 5.1 원안대로 **모집단을 분리**하는 구조로 전환합니다.

(아래는 `PYE_MAX_DLQ_DAY` 기준 1차 버전 — 이후 9번에서 더 나은 타겟으로 교체됩니다)

In [60]:
# ===== Track B 타겟변수 정의: 신용이력 보유군 vs 씬파일러 분리 =====
import pandas as pd

base_ym = '202203'  # 기준 스냅샷 (2021년말 기준. 필요시 202103으로 변경 가능 - 팀 합의 필요)

cols = ['CUST_ID', 'PYE_MAX_DLQ_DAY',
        'PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004',
        'PYE_C18233005', 'PYE_L10210000']

df = pd.read_csv(f'{scan_path}/{base_ym}_통신카드CB결합.csv', usecols=cols)

# 씬파일러 정의 (5개 필드 전부 0)
thin_mask = (
    (df['PYE_C1M210000']==0) & (df['PYE_C18233003']==0) &
    (df['PYE_C18233004']==0) & (df['PYE_C18233005']==0) &
    (df['PYE_L10210000']==0)
)

df['segment'] = thin_mask.map({True: 'thin_filer', False: 'credit_holder'})

# 신용이력 보유군만 분리 -> 여기서 타겟 y 생성
credit_holders = df[df['segment']=='credit_holder'].copy()
credit_holders['y'] = (credit_holders['PYE_MAX_DLQ_DAY'] > 0).astype(int)

thin_filers = df[df['segment']=='thin_filer'].copy()
# 씬파일러는 y를 만들지 않음 - 정답 없이 스코어링만 할 대상

print(f"전체: {len(df):,}명")
print(f"신용이력 보유군: {len(credit_holders):,}명 (학습용)")
print(f"씬파일러: {len(thin_filers):,}명 (적용용, y 없음)")

print(f"\n=== 신용이력 보유군 내 타겟(y) 분포 ===")
print(credit_holders['y'].value_counts())
print(f"양성 비율: {credit_holders['y'].mean()*100:.3f}%")

전체: 330,000명
신용이력 보유군: 288,667명 (학습용)
씬파일러: 41,333명 (적용용, y 없음)

=== 신용이력 보유군 내 타겟(y) 분포 ===
y
0    287468
1      1199
Name: count, dtype: int64
양성 비율: 0.415%


**결과**: 신용이력 보유군 288,667명(양성 1,199명, 0.415%) / 씬파일러 41,333명(적용 전용).

클래스 비율 약 240:1로 다룰 수 있는 수준이지만, 더 나은 후보가 있는지 마지막으로 한 번 더 검토합니다.

## 9. 타겟 후보 시도 ⑤ — `PYE_D1011000C`(1년내발생, 해제포함) — ✅ 최종 채택

`PYE_MAX_DLQ_DAY`는 '미해제'만 잡아서 표본이 매우 작았습니다. `PYE_D1011000C`는 **이미 해소된 연체까지 포함**하는 지표라, 이걸로 대체할 수 있는지 검토합니다.

In [61]:
cols = ['CUST_ID', 'PYE_MAX_DLQ_DAY', 'PYE_D1011000C',
        'PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004',
        'PYE_C18233005', 'PYE_L10210000']

df = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=cols)

thin_mask = (
    (df['PYE_C1M210000']==0) & (df['PYE_C18233003']==0) &
    (df['PYE_C18233004']==0) & (df['PYE_C18233005']==0) &
    (df['PYE_L10210000']==0)
)
credit_holders = df[~thin_mask].copy()

# 두 타겟 정의
credit_holders['y_maxdlq'] = (credit_holders['PYE_MAX_DLQ_DAY'] > 0).astype(int)
credit_holders['y_d1011'] = (credit_holders['PYE_D1011000C'] > 0).astype(int)

print("=== 신용이력 보유군 내 타겟 비교 ===")
print(f"y_maxdlq (미해제 최장연체일수>0) 양성: {credit_holders['y_maxdlq'].sum()}명 ({credit_holders['y_maxdlq'].mean()*100:.3f}%)")
print(f"y_d1011  (1년내발생, 해제포함)   양성: {credit_holders['y_d1011'].sum()}명 ({credit_holders['y_d1011'].mean()*100:.3f}%)")

# 교차표 - 겹치는 정도 확인
print("\n=== 교차표 ===")
print(pd.crosstab(credit_holders['y_maxdlq'], credit_holders['y_d1011'],
                   rownames=['y_maxdlq'], colnames=['y_d1011']))

# y_d1011=1인데 y_maxdlq=0인 사람들 - "연체는 발생했지만 이미 해소된" 케이스
resolved = credit_holders[(credit_holders['y_d1011']==1) & (credit_holders['y_maxdlq']==0)]
print(f"\n발생했지만 이미 해소됨(y_d1011=1, y_maxdlq=0): {len(resolved)}명")

# 씬파일러 쪽도 같이 확인 - 얼마나 항등식에 걸리는지 재확인
thin_df = df[thin_mask].copy()
thin_df['y_maxdlq'] = (thin_df['PYE_MAX_DLQ_DAY'] > 0).astype(int)
thin_df['y_d1011'] = (thin_df['PYE_D1011000C'] > 0).astype(int)
print(f"\n=== 씬파일러 내 비교 (참고용) ===")
print(f"y_maxdlq 양성: {thin_df['y_maxdlq'].sum()}명 ({thin_df['y_maxdlq'].mean()*100:.4f}%)")
print(f"y_d1011  양성: {thin_df['y_d1011'].sum()}명 ({thin_df['y_d1011'].mean()*100:.4f}%)")

=== 신용이력 보유군 내 타겟 비교 ===
y_maxdlq (미해제 최장연체일수>0) 양성: 1199명 (0.415%)
y_d1011  (1년내발생, 해제포함)   양성: 12131명 (4.202%)

=== 교차표 ===
y_d1011        0      1
y_maxdlq               
0         276391  11077
1            145   1054

발생했지만 이미 해소됨(y_d1011=1, y_maxdlq=0): 11077명

=== 씬파일러 내 비교 (참고용) ===
y_maxdlq 양성: 9명 (0.0218%)
y_d1011  양성: 197명 (0.4766%)


In [62]:
# 씬파일러 중 y_d1011=1인 사람들 - 이전 시점(2020년말)에도 씬파일러였는지 확인
cols_prev = ['CUST_ID', 'PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004',
             'PYE_C18233005', 'PYE_L10210000']
df_prev = pd.read_csv(f'{scan_path}/202103_통신카드CB결합.csv', usecols=cols_prev)

thin_prev_mask = (
    (df_prev['PYE_C1M210000']==0) & (df_prev['PYE_C18233003']==0) &
    (df_prev['PYE_C18233004']==0) & (df_prev['PYE_C18233005']==0) &
    (df_prev['PYE_L10210000']==0)
)
thin_prev_ids = set(df_prev.loc[thin_prev_mask, 'CUST_ID'])

thin_d1011_positive = thin_df[thin_df['y_d1011']==1]
always_thin = thin_d1011_positive['CUST_ID'].isin(thin_prev_ids).sum()
print(f"씬파일러(2022, y_d1011=1) 197명 중, 2021년에도 씬파일러였던 사람: {always_thin}명")
print(f"→ 즉 2021년엔 이력이 있었다가 2022년에 씬파일러로 재분류된 사람: {197-always_thin}명")

씬파일러(2022, y_d1011=1) 197명 중, 2021년에도 씬파일러였던 사람: 176명
→ 즉 2021년엔 이력이 있었다가 2022년에 씬파일러로 재분류된 사람: 21명


In [63]:
# 1. 연도 간 안정성 확인 (KCB Score는 86.41%였음 - 비교 기준)
cols_stab = ['CUST_ID', 'PYE_D1011000C']
d21 = pd.read_csv(f'{scan_path}/202103_통신카드CB결합.csv', usecols=cols_stab)
d22 = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=cols_stab)
merged = d21.merge(d22, on='CUST_ID', suffixes=('_21','_22'))

changed = merged['PYE_D1011000C_21'] != merged['PYE_D1011000C_22']
print(f"PYE_D1011000C 연도간 변동 비율: {changed.mean()*100:.2f}%")

# 2. 씬파일러 vs 신용이력보유군 연체율 방향, 연도별 일관성 확인
cols_dir = ['CUST_ID', 'PYE_D1011000C', 'PYE_C1M210000', 'PYE_C18233003',
            'PYE_C18233004', 'PYE_C18233005', 'PYE_L10210000']

for label, path in [('2021','202103'), ('2022','202203')]:
    d = pd.read_csv(f'{scan_path}/{path}_통신카드CB결합.csv', usecols=cols_dir)
    thin = (
        (d['PYE_C1M210000']==0) & (d['PYE_C18233003']==0) &
        (d['PYE_C18233004']==0) & (d['PYE_C18233005']==0) &
        (d['PYE_L10210000']==0)
    )
    thin_rate = (d.loc[thin,'PYE_D1011000C']>0).mean()*100
    gen_rate = (d.loc[~thin,'PYE_D1011000C']>0).mean()*100
    print(f"[{label}] 씬파일러: {thin_rate:.3f}% / 신용이력보유군: {gen_rate:.3f}% / 배율: {thin_rate/gen_rate:.2f}배")

PYE_D1011000C 연도간 변동 비율: 1.41%
[2021] 씬파일러: 0.589% / 신용이력보유군: 4.544% / 배율: 0.13배
[2022] 씬파일러: 0.477% / 신용이력보유군: 4.202% / 배율: 0.11배


**확인 결과 — 두 타겟 비교**

| 비교 항목 | `PYE_MAX_DLQ_DAY` | `PYE_D1011000C` |
|---|---|---|
| 신용이력보유군 양성 | 1,199명(0.415%) | **12,131명(4.202%)** |
| 씬파일러 양성 | 9명(0.0218%) | 197명(0.4766%) |
| 씬파일러 197명 중 '2021년에도 씬파일러'였던 사람 | - | **176명** (21명만 재분류 케이스) |
| 연도간 변동 비율 | 0.45% | **1.41%** (KCB Score 86.41%와 비교 시 매우 안정적) |
| 방향 일관성(2021→2022) | 역전 없음 | **역전 없음** (0.13배 → 0.11배) |

**판정: 채택.** 표본이 10배로 확대되었고, 안정성·방향 일관성 검증을 모두 통과. '해제 포함'이라 정의(스톡)와 타겟(1년간 발생, 플로우) 시점이 어느 정도 분리되어 시도 ③의 항등식 문제도 완화됨.

## 10. 최종 확정 — Track B 타겟변수 및 학습 구조

`PYE_D1011000C`를 타겟으로 최종 확정한 코드입니다. 이후 대안변수(X) 작업은 이 `credit_holders` / `thin_filers` 두 데이터프레임을 기준으로 진행합니다.

In [64]:
base_ym = '202203'

cols = ['CUST_ID', 'PYE_D1011000C',
        'PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004',
        'PYE_C18233005', 'PYE_L10210000']

df = pd.read_csv(f'{scan_path}/{base_ym}_통신카드CB결합.csv', usecols=cols)

thin_mask = (
    (df['PYE_C1M210000']==0) & (df['PYE_C18233003']==0) &
    (df['PYE_C18233004']==0) & (df['PYE_C18233005']==0) &
    (df['PYE_L10210000']==0)
)

df['segment'] = thin_mask.map({True: 'thin_filer', False: 'credit_holder'})

credit_holders = df[df['segment']=='credit_holder'].copy()
credit_holders['y'] = (credit_holders['PYE_D1011000C'] > 0).astype(int)

thin_filers = df[df['segment']=='thin_filer'].copy()

print(f"신용이력 보유군: {len(credit_holders):,}명 (학습용)")
print(f"양성(y=1): {credit_holders['y'].sum():,}명 ({credit_holders['y'].mean()*100:.3f}%)")
print(f"씬파일러: {len(thin_filers):,}명 (적용용)")

신용이력 보유군: 288,667명 (학습용)
양성(y=1): 12,131명 (4.202%)
씬파일러: 41,333명 (적용용)


### 최종 확정 사항 요약

| 항목 | 내용 |
|---|---|
| 씬파일러 정의 | 5개 필드 전부 0 (`PYE_C1M210000`, `PYE_C18233003~005`, `PYE_L10210000`) — 13.37%(2021)/12.53%(2022) |
| 타겟변수(y) | `PYE_D1011000C > 0` (1년 내 연체 발생, 해제포함) |
| 학습 모집단 | 신용이력 보유군 288,667명, 양성 12,131명(4.202%) |
| 적용 모집단 | 씬파일러 41,333명 (정답 없이 스코어링만) |
| 기준 시점 | 202203 (2021년말 기준 스냅샷) |

---

## 11. 다음 단계 (미결)

1. 대안변수(X) 목록 확정 — Track B 사용가능 변수(24개 핵심 / 312개 확장) 중 선정, **씬파일러 정의 5개 필드는 반드시 제외**(순환논리 방지)
2. WOE/IV 계산 → 로지스틱회귀 → XGBoost 순 학습
3. 클래스 비율 약 23:1(양성 4.2%) — 계층화 샘플링 우선 적용, 필요시 XGBoost `scale_pos_weight` 활용
4. 씬파일러 적용 후 등급 분포 결과를 H1(SCORE=0 집단과 무이력 씬파일러 일치)·H5(등급 분포)와 어떻게 연결할지 발표 스토리 정리

## 11. 시점 분리 재설계 검증 — 진짜 forward 구조로 개선

섹션 10까지의 설계는 정의(씬파일러 여부)와 타겟(y)을 **같은 스냅샷(202203)**에서 뽑았습니다. `PYE_D1011000C`가 '해제포함'이라 완전한 항등식은 아니지만, 엄밀히는 '과거 특성으로 미래를 예측'하는 구조가 아니라 '같은 해의 특성과 사건'을 보는 구조였습니다.

이를 개선하기 위해 **정의(X)는 202103(2020년말 기준), 타겟(y)은 202203의 `PYE_D1011000C`(2021년 중 발생)**로 시점을 분리해서 재검증합니다. Track A의 PERF 설계(기준시점 → forward N개월)와 같은 논리 구조입니다.

In [65]:
cols_def = ['CUST_ID', 'PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004',
            'PYE_C18233005', 'PYE_L10210000']
cols_tgt = ['CUST_ID', 'PYE_D1011000C']

# 1. 정의(X 기준): 202103 = 2020년말 기준
df_def = pd.read_csv(f'{scan_path}/202103_통신카드CB결합.csv', usecols=cols_def)

thin_mask_prev = (
    (df_def['PYE_C1M210000']==0) & (df_def['PYE_C18233003']==0) &
    (df_def['PYE_C18233004']==0) & (df_def['PYE_C18233005']==0) &
    (df_def['PYE_L10210000']==0)
)

df_def['segment'] = thin_mask_prev.map({True: 'thin_filer', False: 'credit_holder'})
print(f"[202103 기준] 신용이력 보유군: {(~thin_mask_prev).sum():,}명")
print(f"[202103 기준] 씬파일러: {thin_mask_prev.sum():,}명")

# 2. 타겟(y): 202203 = 2021년 동안 발생한 연체
df_tgt = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=cols_tgt)

# 3. CUST_ID로 매칭 (정의 시점의 사람 + 타겟 시점의 y)
merged = df_def.merge(df_tgt, on='CUST_ID', how='left')

credit_holders_fwd = merged[merged['segment']=='credit_holder'].copy()
credit_holders_fwd['y'] = (credit_holders_fwd['PYE_D1011000C'] > 0).astype(int)

thin_filers_fwd = merged[merged['segment']=='thin_filer'].copy()
thin_filers_fwd['y'] = (thin_filers_fwd['PYE_D1011000C'] > 0).astype(int)  # 참고용

print(f"\n=== [시점분리 버전] 신용이력 보유군 y 분포 ===")
print(f"인원: {len(credit_holders_fwd):,}명")
print(f"양성(y=1): {credit_holders_fwd['y'].sum():,}명 ({credit_holders_fwd['y'].mean()*100:.3f}%)")

print(f"\n=== [시점분리 버전] 씬파일러(202103 기준) 참고용 y ===")
print(f"인원: {len(thin_filers_fwd):,}명")
print(f"양성(y=1): {thin_filers_fwd['y'].sum():,}명 ({thin_filers_fwd['y'].mean()*100:.4f}%)")

# 4. 기존(동일시점) 버전과 비교
print(f"\n=== 비교 ===")
print(f"기존(동일시점 202203 기준): 신용이력보유군 288,667명, 양성 12,131명(4.202%)")
print(f"신규(시점분리 202103→202203): 신용이력보유군 {len(credit_holders_fwd):,}명, 양성 {credit_holders_fwd['y'].sum():,}명({credit_holders_fwd['y'].mean()*100:.3f}%)")

[202103 기준] 신용이력 보유군: 285,890명
[202103 기준] 씬파일러: 44,110명

=== [시점분리 버전] 신용이력 보유군 y 분포 ===
인원: 285,890명
양성(y=1): 12,088명 (4.228%)

=== [시점분리 버전] 씬파일러(202103 기준) 참고용 y ===
인원: 44,110명
양성(y=1): 240명 (0.5441%)

=== 비교 ===
기존(동일시점 202203 기준): 신용이력보유군 288,667명, 양성 12,131명(4.202%)
신규(시점분리 202103→202203): 신용이력보유군 285,890명, 양성 12,088명(4.228%)


**확인 결과 — 기존(동일시점) vs 신규(시점분리) 비교**

| | 기존(동일시점 202203) | 신규(시점분리 202103→202203) |
|---|---|---|
| 신용이력 보유군 | 288,667명 | 285,890명 (거의 동일, -1%) |
| 양성(y=1) | 12,131명(4.202%) | 12,088명(4.228%) (거의 동일) |
| 씬파일러 참고 y | 197명(0.477%) | 240명(0.544%) (소폭 증가) |

표본 수·양성 비율이 거의 그대로 유지되면서, 논리적으로 더 정확한(진짜 forward) 구조를 얻었습니다. 트레이드오프 없이 개선된 것으로 판단해 **이 시점분리 버전을 최종 확정**합니다.

**최종 확정 (수정)**

| 항목 | 내용 |
|---|---|
| 정의(X, 씬파일러 여부) 기준 | 202103 (2020년말 기준) |
| 타겟(y) 기준 | 202203의 `PYE_D1011000C` (2021년 중 발생, 해제포함) |
| 신용이력 보유군 (학습용) | 285,890명, 양성 12,088명(4.228%) |
| 씬파일러 (적용 전용) | 44,110명, 정답 없이 스코어링만 |

### 남은 작업 (다음 단계)
1. **기획서 문구 업데이트/팀 컨펌** — 원안(KCB Score 등급화)에서 `PYE_D1011000C` 이진분류로 전환한 이유 공유
2. **X 변수 선정 시 노출 연동 여부 체크** — 카드 보유가 전제된 파생변수(`CD_USE_AMT` 등)는 씬파일러에게 구조적으로 0이 찍힐 수 있음. 최대한 배제하거나 영향력 별도 확인
3. 대안변수(X) 목록 확정 → WOE/IV → 로지스틱회귀 → XGBoost 순 학습 (기준 시점: X는 202103, y는 202203)